In [ ]:
!pip install -q transformers==4.57.0
!pip install -q accelerate peft trl datasets
!pip install -q einops timm
!pip install -q "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

In [1]:
import os, gc, ctypes, types, shutil
import torch
import numpy as np
import pandas as pd
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ── Auth ──────────────────────────────────────────────────
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
login(hf_token)

# ── Paths ─────────────────────────────────────────────────
PATH_TRAIN    = "REALM/datasets/train"
PATH_TEST     = "REALM/datasets/test"
IMG_DIR_TRAIN = os.path.join(PATH_TRAIN, "images")
IMG_DIR_TEST  = os.path.join(PATH_TEST,  "images")
_MODEL_NAME   = "OpenGVLab/InternVL3-14B"
_OUTPUT_DIR   = "/kaggle/working/internvl3-14b-lora"

# ── Clone repo if needed ──────────────────────────────────
if not os.path.exists("REALM"):
    os.system("git clone https://github.com/LOVISH007/REALM.git")

# ── Memory utility ────────────────────────────────────────
def free_memory(*objects):
    for obj in objects:
        try: del obj
        except: pass
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except: pass

# ── Verify ────────────────────────────────────────────────
print(f"Train dir : {os.path.exists(IMG_DIR_TRAIN)}")
print(f"Test dir  : {os.path.exists(IMG_DIR_TEST)}")
print(f"GPU       : {torch.cuda.get_device_name(0)}")
print(f"Adapter   : {os.path.exists(_OUTPUT_DIR)}  ← must be False")

2026-04-15 21:33:05.186271: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776288785.202547    1569 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776288785.208383    1569 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776288785.220059    1569 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776288785.220075    1569 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776288785.220078    1569 computation_placer.cc:177] computation placer alr

Train dir : True
Test dir  : True
GPU       : NVIDIA H100 80GB HBM3
Adapter   : False  ← must be False


In [2]:
# ── Data ──────────────────────────────────────────────────
df_train = pd.read_csv(os.path.join(PATH_TRAIN, "image_descriptions.csv"))
df_test  = pd.read_csv(os.path.join(PATH_TEST,  "image_descriptions.csv"))
print(f"Train: {len(df_train)} | Test: {len(df_test)}")

PROMPT = """You are an expert at determining if the given image is AI-GENERATED or not. 
Answer YES if you feel it is AI-GENERATED and answer NO if you feel it is no and answer Somewhat if you are not sure. 

FORMAT TO ANSWER : 
Reply in exactly this format and respond in English only:
Yes/No/Somewhat, <one sentence identifying the key visual clue that is key behind your answer>.

Examples:
- No,there is nothing unrealistic in this image,  the almonds are arranged in a heart shape but the layout looks natural and plausible.
- Yes , the image is AI generated, the roots and soil mound look oddly artificial and the people in the background are unnaturally repetitive.
- Somewhat, the hat appears unusually large and its integration with the fur looks slightly unnatural."""

# ── Image preprocessing ───────────────────────────────────
transform = T.Compose([
    T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    T.Resize((448, 448), interpolation=InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

NUM_IMAGE_TOKENS  = 256
image_placeholder = "<img>" + "<IMG_CONTEXT>" * NUM_IMAGE_TOKENS + "</img>"

# ── Load tokenizer ────────────────────────────────────────
print("Loading tokenizer...")
_tokenizer = AutoTokenizer.from_pretrained(
    _MODEL_NAME, trust_remote_code=True, token=hf_token
)
print("Tokenizer loaded ✓")

# ── Load model ────────────────────────────────────────────
print("Loading model...")
_model = AutoModel.from_pretrained(
    _MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    token=hf_token,
    device_map="auto",
)
_model.config.use_cache = False
_model.gradient_checkpointing_enable()
_model.enable_input_require_grads()
print("Model loaded ✓")

# ── Set image context token BEFORE LoRA ───────────────────
img_context_token_id = _tokenizer.convert_tokens_to_ids("<IMG_CONTEXT>")
_model.img_context_token_id = img_context_token_id
print(f"img_context_token_id: {img_context_token_id} ✓")

# ── Fresh LoRA ────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
_model = get_peft_model(_model, lora_config)
_model.base_model.model.img_context_token_id = img_context_token_id
_model.print_trainable_parameters()
print("Fresh LoRA applied ✓")

Train: 510 | Test: 90
Loading tokenizer...
Tokenizer loaded ✓
Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Model loaded ✓
img_context_token_id: 151667 ✓
trainable params: 68,812,800 || all params: 15,186,069,504 || trainable%: 0.4531
Fresh LoRA applied ✓


In [5]:
# ── Build dataset ─────────────────────────────────────────
def build_hf_dataset(df, img_dir):
    records = []
    for _, row in df.iterrows():
        image = Image.open(os.path.join(img_dir, row["filename"])).convert("RGB")
        messages = [
            {"role": "user",      "content": f"{image_placeholder}\n{PROMPT}"},
            {"role": "assistant", "content": str(row["description"]).strip()},
        ]
        text = _tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        records.append({"text": text, "pixel_values": transform(image)})
    print(f"Dataset built: {len(records)} samples ✓")
    return Dataset.from_list(records)

train_dataset = build_hf_dataset(df_train, IMG_DIR_TRAIN)

# ── Fixed collator ────────────────────────────────────────
ASSISTANT_TOKEN     = "<|im_start|>assistant\n"
assistant_token_ids = _tokenizer.encode(ASSISTANT_TOKEN, add_special_tokens=False)
assist_len          = len(assistant_token_ids)
assist_tensor       = torch.tensor(assistant_token_ids)

def collate_fn(examples):
    texts        = [ex["text"]         for ex in examples]
    pixel_values = [ex["pixel_values"] for ex in examples]

    batch = _tokenizer(
        texts, return_tensors="pt", padding=True,
        truncation=True, max_length=1024,
    )

    input_ids      = batch["input_ids"]
    attention_mask = batch["attention_mask"]
    labels         = input_ids.clone()

    for i in range(input_ids.shape[0]):
        seq        = input_ids[i]
        mask_until = 0
        for j in range(len(seq) - assist_len + 1):
            if torch.equal(seq[j:j+assist_len], assist_tensor):
                mask_until = j + assist_len
                break
        labels[i, :mask_until] = -100
        labels[i, attention_mask[i] == 0] = -100

    pixel_values_tensor = torch.stack([
        torch.tensor(pv) if not isinstance(pv, torch.Tensor) else pv
        for pv in pixel_values
    ]).to(torch.bfloat16)

    batch["pixel_values"] = pixel_values_tensor
    batch["labels"]       = labels
    batch["image_flags"]  = torch.ones(input_ids.shape[0], 1, dtype=torch.long)
    return batch

# ── Verify collator ───────────────────────────────────────
s = collate_fn([train_dataset[0]])
non_masked = s["input_ids"][0][s["labels"][0] != -100]
print(f"Tokens in loss : {(s['labels'][0] != -100).sum().item()}")
print(f"Training on    : '{_tokenizer.decode(non_masked)}'")

# ── Forward patch ─────────────────────────────────────────
_true_original_forward = type(_model.base_model.model).forward

def _patched_forward(self, *args, **kwargs):
    kwargs.pop("inputs_embeds", None)
    kwargs.pop("num_items_in_batch", None)
    return _true_original_forward(self, *args, **kwargs)

_model.base_model.model.forward = types.MethodType(
    _patched_forward, _model.base_model.model
)
print("Forward patched ✓")

# ── Trainer ───────────────────────────────────────────────
class InternVLTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(
            pixel_values   = inputs.get("pixel_values"),
            input_ids      = inputs.get("input_ids"),
            attention_mask = inputs.get("attention_mask"),
            labels         = inputs.get("labels"),
            image_flags    = inputs.get("image_flags"),
        )
        return (outputs.loss, outputs) if return_outputs else outputs.loss

training_args = TrainingArguments(
    output_dir                  = _OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 16,
    learning_rate               = 2e-5,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    bf16                        = True,
    fp16                        = False,
    gradient_checkpointing      = True,
    dataloader_num_workers      = 2,
    logging_steps               = 10,
    save_strategy               = "epoch",
    save_total_limit            = 1,
    optim                       = "adamw_torch",
    report_to                   = "none",
    remove_unused_columns       = False,
    label_names                 = ["labels"],
)

trainer = InternVLTrainer(
    model            = _model,
    args             = training_args,
    train_dataset    = train_dataset,
    data_collator    = collate_fn,
    processing_class = _tokenizer,
)

print("\nStarting clean training...")
print(f"  Steps per epoch : {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print(f"  Total steps     : {(len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)) * training_args.num_train_epochs}")

trainer.train()

trainer.save_model(_OUTPUT_DIR)
_tokenizer.save_pretrained(_OUTPUT_DIR)
print(f"\nLoRA adapter saved to {_OUTPUT_DIR} ✓")

Dataset built: 510 samples ✓


The model is already on multiple devices. Skipping the move to device specified in `args`.


Tokens in loss : 32
Training on    : 'Somewhat. The hat appears unusually large and may not sit naturally on the dog's head; the integration between the hat and fur looks slightly unnatural.<|im_end|>
'
Forward patched ✓

Starting clean training...
  Steps per epoch : 31
  Total steps     : 93


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
10,21.788700
20,19.063900
30,18.655200
40,18.456400
50,17.710900
60,17.007100
70,17.862400
80,16.795100
90,17.354100


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the


LoRA adapter saved to /kaggle/working/internvl3-14b-lora ✓


In [6]:
from peft import PeftModel

# ── Load base + adapter (NO merge) ───────────────────────
print("Loading fine-tuned model...")
_base = AutoModel.from_pretrained(
    _MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    token=hf_token,
    device_map="auto",
)
_ft_model = PeftModel.from_pretrained(_base, _OUTPUT_DIR)
_ft_model.eval()
print("Fine-tuned model loaded ✓")

def run_internvl_inference(df, img_dir, model, tokenizer, model_col, batch_size=1):
    all_responses = []
    for start in range(0, len(df), batch_size):
        batch_df = df.iloc[start:start+batch_size]
        responses = []
        for _, row in batch_df.iterrows():
            img_path     = os.path.join(img_dir, row["filename"])
            image        = Image.open(img_path).convert("RGB")
            pixel_values = transform(image).unsqueeze(0).to(
                next(model.parameters()).device
            ).to(torch.bfloat16)

            response, _ = model.chat(
                tokenizer         = tokenizer,
                pixel_values      = pixel_values,
                question          = PROMPT,
                generation_config = dict(max_new_tokens=100, do_sample=False),
                history           = None,
                return_history    = True,
            )
            responses.append(response)

        all_responses.extend(responses)
        print(f"  {min(start+batch_size, len(df))}/{len(df)} done", end="\r")

    df = df.copy()
    df[model_col] = all_responses
    return df

print("\nRunning fine-tuned inference on full test set...")
df_test = run_internvl_inference(
    df_test, IMG_DIR_TEST, _ft_model, _tokenizer,
    model_col="InternVL3-14B-FT",
)
df_test.to_csv("/kaggle/working/results.csv", index=False)
print("\nFine-tuned inference done ✓")
print("\nSample outputs (head 6):")
for i, row in df_test.head(6).iterrows():
    print(f"  GT : {row['description']}")
    print(f"  FT : {row['InternVL3-14B-FT']}")
    print()

Loading fine-tuned model...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Fine-tuned model loaded ✓

Running fine-tuned inference on full test set...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  90/90 done
Fine-tuned inference done ✓

Sample outputs (head 6):
  GT : Somewhat. The front wheel appears distorted and unnaturally angled, and the reflection on the car’s door is inconsistent with the background, making the image look slightly unrealistic.
  FT : Somewhat. The car's wheels and side mirrors appear slightly distorted and unnatural, and the transition between the car and the background is not smooth, suggesting possible digital manipulation.

  GT : No, there is nothing unrealistic in this image. The leaf and the background both appear natural and visually consistent.
  FT : No, there is nothing unrealistic in this image. The leaf and background appear natural and well-rendered.

  GT : No, there is nothing unrealistic in this image. All objects, people, and buildings appear normal and proportionate, with smooth transitions and no obvious distortions.
  FT : Somewhat. The transition between the sidewalk and the street appears unnatural, with an abrupt change in texture

In [7]:
# ── Free fine-tuned model ─────────────────────────────────
free_memory(_ft_model, _base)
print("Memory freed ✓")

# ── Load clean base model ─────────────────────────────────
print("Loading base model for zero-shot inference...")
_base_model = AutoModel.from_pretrained(
    _MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    token=hf_token,
    device_map="auto",
)
_base_model.eval()
print("Base model loaded ✓")

# ── Run zero-shot inference ───────────────────────────────
def run_internvl_inference(df, img_dir, model, tokenizer, model_col, batch_size=1):
    all_responses = []
    for start in range(0, len(df), batch_size):
        batch_df = df.iloc[start:start+batch_size]
        responses = []
        for _, row in batch_df.iterrows():
            img_path     = os.path.join(img_dir, row["filename"])
            image        = Image.open(img_path).convert("RGB")
            pixel_values = transform(image).unsqueeze(0).to(
                next(model.parameters()).device
            ).to(torch.bfloat16)

            response, _ = model.chat(
                tokenizer         = tokenizer,
                pixel_values      = pixel_values,
                question          = PROMPT,
                generation_config = dict(max_new_tokens=100, do_sample=False),
                history           = None,
                return_history    = True,
            )
            responses.append(response)

        all_responses.extend(responses)
        print(f"  {min(start+batch_size, len(df))}/{len(df)} done", end="\r")

    df = df.copy()
    df[model_col] = all_responses
    return df

print("\nRunning zero-shot inference on 90 test images...")
df_test = run_internvl_inference(
    df_test, IMG_DIR_TEST, _base_model, _tokenizer,
    model_col="InternVL3-14B",
)
df_test.to_csv("/kaggle/working/results.csv", index=False)
print("\nZero-shot done ✓")
print("\nSample outputs:")
for i, row in df_test.head(3).iterrows():
    print(f"  GT       : {row['description']}")
    print(f"  Baseline : {row['InternVL3-14B']}")
    print()

Memory freed ✓
Loading base model for zero-shot inference...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Base model loaded ✓

Running zero-shot inference on 90 test images...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  90/90 done
Zero-shot done ✓

Sample outputs:
  GT       : Somewhat. The front wheel appears distorted and unnaturally angled, and the reflection on the car’s door is inconsistent with the background, making the image look slightly unrealistic.
  Baseline : No, the image appears realistic with natural reflections and details typical of a high-quality photograph of a car.

  GT       : No, there is nothing unrealistic in this image. The leaf and the background both appear natural and visually consistent.
  Baseline : No, the leaf and background appear natural with realistic lighting and texture.

  GT       : No, there is nothing unrealistic in this image. All objects, people, and buildings appear normal and proportionate, with smooth transitions and no obvious distortions.
  Baseline : No, the image appears to be a realistic urban scene with no obvious signs of artificiality or anomalies.



In [9]:
from peft import PeftModel

# ── Force clear everything ────────────────────────────────
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

# ── Load base + adapter cleanly ───────────────────────────
print("Loading fine-tuned model...")
_base = AutoModel.from_pretrained(
    _MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    token=hf_token,
    device_map={"": 0},   # ← force everything to GPU 0, no CPU offload
)
_ft_model = PeftModel.from_pretrained(_base, _OUTPUT_DIR)
_ft_model.eval()
print("Fine-tuned model loaded ✓")

# ── Run FT inference ──────────────────────────────────────
print("\nRunning fine-tuned inference on 90 test images...")
df_test = run_internvl_inference(
    df_test, IMG_DIR_TEST, _ft_model, _tokenizer,
    model_col="InternVL3-14B-FT",
)
df_test.to_csv("/kaggle/working/results.csv", index=False)
print("\nFT inference done ✓")
print("\nSample outputs:")
for i, row in df_test.head(3).iterrows():
    print(f"  GT : {row['description']}")
    print(f"  FT : {row['InternVL3-14B-FT']}")
    print()

VRAM free: 35.0 GB
Loading fine-tuned model...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Fine-tuned model loaded ✓

Running fine-tuned inference on 90 test images...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  90/90 done
FT inference done ✓

Sample outputs:
  GT : Somewhat. The front wheel appears distorted and unnaturally angled, and the reflection on the car’s door is inconsistent with the background, making the image look slightly unrealistic.
  FT : Somewhat. The car's wheels and side mirrors appear slightly distorted and unnatural, and the transition between the car and the background is not smooth, suggesting possible digital manipulation.

  GT : No, there is nothing unrealistic in this image. The leaf and the background both appear natural and visually consistent.
  FT : No, there is nothing unrealistic in this image. The leaf and background appear natural and well-rendered.

  GT : No, there is nothing unrealistic in this image. All objects, people, and buildings appear normal and proportionate, with smooth transitions and no obvious distortions.
  FT : Somewhat. The transition between the sidewalk and the street appears unnatural, with an abrupt change in texture and color, and t

In [10]:
from google import genai
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime

# ── Gemini client ─────────────────────────────────────────
gemini_api_key = user_secrets.get_secret("GEMINI_API_KEY")
client = genai.Client(api_key=gemini_api_key)

# ── Pydantic schemas ──────────────────────────────────────
class ScoreDistribution(BaseModel):
    prob_1: float
    prob_2: float
    prob_3: float
    prob_4: float
    prob_5: float

class EvalResult(BaseModel):
    score_distribution: ScoreDistribution
    reasoning: str

class EvalResultList(BaseModel):
    results: List[EvalResult]

def compute_weighted_score(dist):
    return 1*dist.prob_1 + 2*dist.prob_2 + 3*dist.prob_3 + 4*dist.prob_4 + 5*dist.prob_5

GEVAL_PROMPT = """You are an expert evaluator assessing how well a VLM explanation aligns with a human reference explanation about whether an image is real or AI-generated.

## Core Principle
You are checking SEMANTIC OVERLAP — does the VLM identify the same or equivalent visual artifacts as the human? Extra observations by the VLM are FINE but should incur a small penalty.

## Evaluation Steps (follow in order)
1. Extract the human's VERDICT: Real / AI-generated / Uncertain.
2. Extract the VLM's VERDICT.
3. Check VERDICT AGREEMENT:
   - Same verdict -> proceed to step 4.
   - Opposite verdicts (one says Real, other says AI) -> score 1-2, stop.
   - One is Uncertain, other is definitive -> mild penalty only.
4. Count how many distinct subjects/regions the human mentions.
   - If SINGLE subject (e.g. only a giraffe, only a car): the VLM MUST identify the correct artifact on that subject.
   - If MULTIPLE subjects/regions: partial credit allowed when VLM identifies some but not all.
5. Check if VLM covers those core artifacts (exact or semantically equivalent).
6. Extra artifacts mentioned by VLM but NOT in human -> small penalty (~0.3-0.5 weighted score).

## Scoring Guide (1-5)
- 5: Verdicts agree, all core artifacts matched, no extra artifacts.
- 4: Verdicts agree, core artifacts matched, minor extra artifacts OR one small miss.
- 3: Verdicts agree, VLM identifies wrong artifact on same subject OR misses one of multiple core artifacts.
- 2: Verdicts agree but VLM misses all core artifacts, OR wrong artifact on single-subject image, OR verdicts partially conflict.
- 1: Verdicts directly contradict (Real vs AI-generated).

## Output per pair
- score_distribution: probabilities over scores 1-5 (must sum to 1.0)
- reasoning: one sentence on verdict match + artifact overlap quality."""

def geval_vlm_responses(human_responses, vlm_responses, batch_size=10):
    all_results = []
    for i in range(0, len(human_responses), batch_size):
        h_batch = human_responses[i:i+batch_size]
        v_batch = vlm_responses[i:i+batch_size]
        pairs_text = "\n\n".join(
            f"--- Pair {j+1} ---\nHuman: {h}\nVLM: {v}"
            for j, (h, v) in enumerate(zip(h_batch, v_batch))
        )
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=f"{GEVAL_PROMPT}\n\nEvaluate these {len(h_batch)} pairs:\n\n{pairs_text}",
            config={
                "response_mime_type": "application/json",
                "response_json_schema": EvalResultList.model_json_schema(),
                "max_output_tokens": 10000,
            },
        )
        parsed = EvalResultList.model_validate_json(response.text).results
        for batch_i, r in enumerate(parsed):
            all_results.append({
                "index": i + batch_i,
                "score_distribution": r.score_distribution,
                "weighted_score": compute_weighted_score(r.score_distribution),
                "reasoning": r.reasoning,
            })
        print(f"  Batch {i//batch_size+1} done — {len(parsed)} pairs scored")
    return all_results

def vllm_score(results):
    scores = [r["weighted_score"] for r in results]
    return float(np.mean(scores)), float(np.var(scores))

# ── Report helpers ────────────────────────────────────────
REPORT_PATH         = "/kaggle/working/vlm_geval_report.md"
TOP_N_REPORT_IMAGES = 5
REPORT_MODEL_ORDER  = ["InternVL3-14B", "InternVL3-14B-FT"]
report_state        = {}

def _escape(text):
    return str(text).replace("\n", " ").replace("|", "\\|").strip() if text else ""

def write_markdown_report():
    lines = [
        "# VLM GEval Image-Centric Report", "",
        f"_Last updated: {datetime.now().isoformat(timespec='seconds')}_", "",
    ]
    for image_idx in sorted(report_state.keys()):
        info = report_state[image_idx]
        lines += [
            f"## Image {image_idx+1}: {info['filename']}", "",
            f"**Actual answer:** {_escape(info['actual_answer'])}", "",
            "| Model | Model Answer | Image GEval Score | Model Overall GEval | Model Variance | GEval Reasoning |",
            "|---|---|---:|---:|---:|---|",
        ]
        for model_name in REPORT_MODEL_ORDER:
            row = info["rows"].get(model_name)
            if row is None:
                lines.append(f"| {model_name} | - | - | - | - | - |")
            else:
                lines.append(
                    f"| {_escape(row['model'])} | "
                    f"{_escape(row['model_answer'])} | "
                    f"{row['image_geval_score']:.3f} | "
                    f"{row['model_overall_geval']:.3f} | "
                    f"{row['model_variance']:.3f} | "
                    f"{_escape(row['geval_reasoning'])} |"
                )
        lines.append("")
    with open(REPORT_PATH, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

def init_markdown_report(df, img_dir, n=TOP_N_REPORT_IMAGES):
    global report_state
    report_state = {}
    for idx, row in df.head(n).iterrows():
        report_state[int(idx)] = {
            "filename":      row["filename"],
            "actual_answer": row["description"],
            "image_path":    os.path.join(img_dir, row["filename"]),
            "rows":          {},
        }
    write_markdown_report()
    print(f"Report initialized ✓")

def update_report_for_model(df, results, model_col, model_name,
                             avg_score, variance, n=TOP_N_REPORT_IMAGES):
    results_by_index = {int(r["index"]): r for r in results}
    for idx, row in df.head(n).iterrows():
        idx = int(idx)
        if idx not in report_state:
            continue
        result_row = results_by_index.get(idx)
        if result_row is None:
            continue
        report_state[idx]["rows"][model_name] = {
            "model":               model_name,
            "model_answer":        row[model_col],
            "image_geval_score":   float(result_row["weighted_score"]),
            "model_overall_geval": float(avg_score),
            "model_variance":      float(variance),
            "geval_reasoning":     result_row["reasoning"],
        }
    write_markdown_report()
    print(f"Report updated for {model_name} ✓")

# ── Init report ───────────────────────────────────────────
init_markdown_report(df_test, IMG_DIR_TEST)

# ── G-Eval baseline ───────────────────────────────────────
print("\n=== G-Eval: InternVL3-14B (zero-shot) ===")
baseline_results = geval_vlm_responses(
    list(df_test["description"]),
    list(df_test["InternVL3-14B"]),
    batch_size=10,
)
baseline_avg, baseline_var = vllm_score(baseline_results)
print(f"\nBaseline Average Score : {baseline_avg:.3f}")
print(f"Baseline Variance      : {baseline_var:.3f}")

update_report_for_model(
    df_test, baseline_results,
    model_col="InternVL3-14B",
    model_name="InternVL3-14B",
    avg_score=baseline_avg,
    variance=baseline_var,
)

# ── G-Eval fine-tuned ─────────────────────────────────────
print("\n=== G-Eval: InternVL3-14B-FT ===")
ft_results = geval_vlm_responses(
    list(df_test["description"]),
    list(df_test["InternVL3-14B-FT"]),
    batch_size=10,
)
ft_avg, ft_var = vllm_score(ft_results)
print(f"\nFine-tuned Average Score : {ft_avg:.3f}")
print(f"Fine-tuned Variance      : {ft_var:.3f}")
print(f"Baseline Average Score   : {baseline_avg:.3f}")
print(f"Delta                    : {ft_avg - baseline_avg:+.3f}")

update_report_for_model(
    df_test, ft_results,
    model_col="InternVL3-14B-FT",
    model_name="InternVL3-14B-FT",
    avg_score=ft_avg,
    variance=ft_var,
)

# ── Save everything ───────────────────────────────────────
df_test.to_csv("/kaggle/working/results.csv", index=False)
print(f"\nreport  → {REPORT_PATH} ✓")
print(f"results → /kaggle/working/results.csv ✓")

Report initialized ✓

=== G-Eval: InternVL3-14B (zero-shot) ===
  Batch 1 done — 10 pairs scored
  Batch 2 done — 10 pairs scored
  Batch 3 done — 10 pairs scored
  Batch 4 done — 10 pairs scored
  Batch 5 done — 10 pairs scored
  Batch 6 done — 10 pairs scored
  Batch 7 done — 10 pairs scored
  Batch 8 done — 10 pairs scored
  Batch 9 done — 10 pairs scored

Baseline Average Score : 3.102
Baseline Variance      : 2.771
Report updated for InternVL3-14B ✓

=== G-Eval: InternVL3-14B-FT ===
  Batch 1 done — 10 pairs scored
  Batch 2 done — 10 pairs scored
  Batch 3 done — 10 pairs scored
  Batch 4 done — 10 pairs scored
  Batch 5 done — 10 pairs scored
  Batch 6 done — 10 pairs scored
  Batch 7 done — 10 pairs scored
  Batch 8 done — 10 pairs scored
  Batch 9 done — 10 pairs scored

Fine-tuned Average Score : 6.111
Fine-tuned Variance      : 434.786
Baseline Average Score   : 3.102
Delta                    : +3.009
Report updated for InternVL3-14B-FT ✓

report  → /kaggle/working/vlm_geval

In [11]:
def geval_vlm_responses_safe(human_responses, vlm_responses, batch_size=10):
    all_results = []
    for i in range(0, len(human_responses), batch_size):
        h_batch = human_responses[i:i+batch_size]
        v_batch = vlm_responses[i:i+batch_size]
        pairs_text = "\n\n".join(
            f"--- Pair {j+1} ---\nHuman: {h}\nVLM: {v}"
            for j, (h, v) in enumerate(zip(h_batch, v_batch))
        )
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=f"{GEVAL_PROMPT}\n\nEvaluate these {len(h_batch)} pairs:\n\n{pairs_text}",
            config={
                "response_mime_type": "application/json",
                "response_json_schema": EvalResultList.model_json_schema(),
                "max_output_tokens": 10000,
            },
        )
        parsed = EvalResultList.model_validate_json(response.text).results
        for batch_i, r in enumerate(parsed):
            dist  = r.score_distribution
            total = dist.prob_1 + dist.prob_2 + dist.prob_3 + dist.prob_4 + dist.prob_5

            # Normalize if probs don't sum to 1
            if abs(total - 1.0) > 0.01 and total > 0:
                dist.prob_1 /= total
                dist.prob_2 /= total
                dist.prob_3 /= total
                dist.prob_4 /= total
                dist.prob_5 /= total

            score = compute_weighted_score(dist)

            # Clamp to valid range
            score = max(1.0, min(5.0, score))

            all_results.append({
                "index":              i + batch_i,
                "score_distribution": dist,
                "weighted_score":     score,
                "reasoning":          r.reasoning,
            })
        print(f"  Batch {i//batch_size+1} done — {len(parsed)} pairs scored")
    return all_results

# ── Re-run G-Eval for FT only ─────────────────────────────
print("=== G-Eval: InternVL3-14B-FT (fixed) ===")
ft_results = geval_vlm_responses_safe(
    list(df_test["description"]),
    list(df_test["InternVL3-14B-FT"]),
    batch_size=10,
)
ft_avg, ft_var = vllm_score(ft_results)
print(f"\nFine-tuned Average Score : {ft_avg:.3f}")
print(f"Fine-tuned Variance      : {ft_var:.3f}")
print(f"Baseline Average Score   : {baseline_avg:.3f}")
print(f"Delta                    : {ft_avg - baseline_avg:+.3f}")

# ── Update report ─────────────────────────────────────────
update_report_for_model(
    df_test, ft_results,
    model_col   = "InternVL3-14B-FT",
    model_name  = "InternVL3-14B-FT",
    avg_score   = ft_avg,
    variance    = ft_var,
)

df_test.to_csv("/kaggle/working/results.csv", index=False)
print(f"\nreport  → {REPORT_PATH} ✓")
print(f"results → /kaggle/working/results.csv ✓")

=== G-Eval: InternVL3-14B-FT (fixed) ===
  Batch 1 done — 10 pairs scored
  Batch 2 done — 10 pairs scored
  Batch 3 done — 10 pairs scored
  Batch 4 done — 10 pairs scored
  Batch 5 done — 10 pairs scored
  Batch 6 done — 10 pairs scored
  Batch 7 done — 10 pairs scored
  Batch 8 done — 10 pairs scored
  Batch 9 done — 10 pairs scored

Fine-tuned Average Score : 3.906
Fine-tuned Variance      : 1.362
Baseline Average Score   : 3.102
Delta                    : +0.804
Report updated for InternVL3-14B-FT ✓

report  → /kaggle/working/vlm_geval_report.md ✓
results → /kaggle/working/results.csv ✓


In [12]:
from peft import PeftModel

# Push only the LoRA adapter
_ft_model.push_to_hub("pranshu2004/internvl3-14b-realm-lora")
_tokenizer.push_to_hub("pranshu2004/internvl3-14b-realm-lora")
print("Pushed to HuggingFace ✓")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to HuggingFace ✓
